# 🧠 Conditional VAE: Teaching Machines to Generate on Demand

In-Class Exercise

---

## 🎯 The Big Picture: What Is a Conditional VAE?

In the previous exercise, you built an **Autoencoder** — a network that compresses images into a compact representation and reconstructs them. Think of it as a **photocopier**: it can reproduce what it sees, but it can’t create anything new.

Now we upgrade to a **Conditional Variational Autoencoder (CVAE)**. The key differences:

| | Autoencoder | VAE | Conditional VAE |
|---|---|---|---|
| Bottleneck | Fixed vector | Probability distribution | Probability distribution |
| Can generate? | ❌ No | ✅ Yes (random) | ✅ Yes (**controlled**) |
| Analogy | 📋 Photocopier | 🎨 Artist painting randomly | 🎨 Artist following a brief |

**The analogy:** Imagine an art studio with two specialists:
- 🧐 **The Critic** (Encoder): studies a painting and summarises its *style* — how thick the strokes are, the slant, the size — into a short description.
- 🎨 **The Artist** (Decoder): receives a style description **plus a written brief** (“paint a 7”) and creates a new painting matching both.

The **brief** is the *condition* — in our case, a digit label. This lets us tell the network **what** to generate while the latent space captures **how** it looks.

### 📋 What We’ll Do Today
1. 📦 Explore the dataset (MNIST with one-hot labels)
2. 🤖 Build a Conditional VAE from scratch in PyTorch
3. 🏋️ Train it on MNIST with digit labels as conditions
4. 🔍 Reconstruct images and measure quality
5. ✨ **Generate new digits on demand** — the key CVAE superpower
6. 🌍 Explore the latent space
7. 🔀 Interpolate between digits in latent space
8. 🎛️ Experiment with the beta hyperparameter

In [ ]:
# @title ⚙️ Setup {display-mode: "form"}

# @markdown The cells below clone the course repo and import all helper functions. You don't need to understand what's in these files  just run them and move on!
# -- Step 1: Clone the course repo ----------------------------------------
%cd /content
![ ! -d 'coding-exercises' ] && git clone https://github.com/eth-bmai-fs26/coding-exercises.git
%cd coding-exercises
!git checkout main
%cd "CVAE CX"

# -- Step 2: Import helpers ------------------------------------------------
from utils import *      # training loops, plotting helpers, DEVICE
from models import *     # MNISTClassifier
from data import *       # load_mnist(), to_one_hot()

setup()                  # confirms device (CPU / GPU)

---
## 📦 Part 1: Exploring the Dataset

### Same Data, New Twist

We use the same **MNIST dataset** as before — 70,000 handwritten digit images (0–9), each 28×28 pixels. But this time, **the digit labels are part of the input**, not just for evaluation.

We convert each label to a **one-hot vector**: digit 3 becomes `[0, 0, 0, 1, 0, 0, 0, 0, 0, 0]`. This is how we tell the CVAE which digit we’re working with.

In [ ]:
X_train, y_train, X_test, y_test = load_mnist()

# Convert labels to one-hot encoding
y_train_oh = to_one_hot(y_train)
y_test_oh  = to_one_hot(y_test)

print(f"\nOne-hot label shape: {y_train_oh.shape}")
print(f"Example: digit {y_train[0]} -> {y_train_oh[0]}")

In [ ]:
# @title 👀 Let's look at some examples grouped by digit {display-mode: "form"}

fig, axes = plt.subplots(2, 10, figsize=(20, 4))
fig.suptitle("Sample digits from MNIST", fontsize=14, fontweight='bold')
for d in range(10):
    mask = y_train == d
    for row in range(2):
        idx = np.where(mask)[0][row]
        axes[row, d].imshow(X_train[idx], cmap='gray')
        axes[row, d].axis('off')
    axes[0, d].set_title(str(d), fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🤖 Part 2: Understanding the CVAE Architecture

### From Autoencoder to VAE

A regular autoencoder maps each image to a **fixed point** in latent space. This is great for compression, but makes generation difficult — you don’t know which points in latent space produce valid images.

A **Variational Autoencoder (VAE)** solves this by mapping each image to a **probability distribution** (described by a mean $\mu$ and variance $\sigma^2$). During training, we sample from this distribution, which forces the latent space to be **smooth and continuous** — nearby points decode to similar images.

### 🎲 The Reparametrization Trick

We can’t backpropagate through random sampling. The trick: instead of sampling $z \sim \mathcal{N}(\mu, \sigma^2)$ directly, we compute:

$$z = \mu + \sigma \odot \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, I)$$

The randomness is now in $\varepsilon$ (which doesn’t depend on model parameters), so gradients flow through $\mu$ and $\sigma$ normally.

### 🔗 From VAE to Conditional VAE

In a standard VAE, you sample $z$ and decode — but you can’t control **what** comes out. A **CVAE** adds the condition $c$ (the digit label) at two points:

1. 📥 **Encoder input**: the image channels are concatenated with the label (broadcast to spatial dimensions). This way the encoder learns to extract *style* (not content, since content is given by the label).
2. 📤 **Decoder input**: the sampled $z$ is concatenated with the label. This tells the decoder *what* to generate.

### 📉 The Loss Function (ELBO)

The CVAE is trained to maximise the **Evidence Lower Bound (ELBO)**:

$$\mathcal{L} = \underbrace{\mathbb{E}_{q(z|x,c)}[\log p(x|z,c)]}_{\text{Reconstruction (BCE)}} - \underbrace{\beta \cdot D_{KL}\big(q(z|x,c) \| p(z)\big)}_{\text{KL Divergence}}$$

- 🔄 **Reconstruction loss (BCE)**: how well does the output match the input?
- 📏 **KL Divergence**: how close is the learned distribution to a standard Gaussian $\mathcal{N}(0, I)$?
- ⚖️ **$\beta$**: controls the trade-off. Higher $\beta$ means a more regular latent space but blurrier reconstructions.

In [ ]:
# @title 🧩 Understanding Check: Why add the KL term? { display-mode: "form" }

# @markdown **Question:** What would happen if we trained with only the reconstruction loss (no KL term)?
# @markdown
# @markdown A) The model would fail to learn anything  
# @markdown B) The latent space would collapse — all images map to the same point  
# @markdown C) The latent space would have "holes" — sampling random z would produce garbage  
# @markdown D) The model would generate perfect images

# @markdown ---
# @markdown **Answer:** C. Without KL regularisation, the encoder can spread images
# @markdown arbitrarily across latent space. The decoder memorises these specific locations,
# @markdown but random samples would land in "empty" regions that never appeared during training.

---
### 🎯 TODO 1 — Complete the ConvCVAE class

Below is the architecture of our Conditional VAE. The `__init__` defines all layers. Your task is to fill in the methods:

**`conditional_input(self, x, label)`** — Prepare the encoder input:
1. Reshape `label` from `(B, 10)` to `(B, 10, 1, 1)`
2. Expand to `(B, 10, 28, 28)` using `.expand()`
3. Concatenate with `x` along the channel dimension → `(B, 11, 28, 28)`

**`reparametrize(self, mu, log_var)`** — The reparametrization trick:
1. Compute `std = exp(0.5 * log_var)`
2. Sample `eps` from $\mathcal{N}(0, I)$ with the same shape as `std`
3. Return `mu + std * eps`

**`encode(self, x, label)`** — Encoder forward pass:
1. Build conditional input
2. Pass through `self.encoder_conv`, flatten, then through `self.fc_mu` and `self.fc_log_var`
3. Return `mu, log_var`

**`decode(self, z, label)`** — Decoder forward pass:
1. Concatenate `z` and `label` along dim=1 → `(B, latent_dim + 10)`
2. Pass through `self.decoder_fc`, reshape to `(B, 128, 7, 7)`, then through `self.decoder_conv`

**`forward(self, x, label)`** — Full CVAE pass:
1. Encode → reparametrize → decode
2. Return `recon_x, mu, log_var`

In [ ]:
class ConvCVAE(nn.Module):
    """
    Convolutional Conditional VAE for MNIST.
    Encoder: (11, 28, 28) -> mu, log_var  (latent_dim each)
    Decoder: (latent_dim + 10) -> (1, 28, 28)
    """
    def __init__(self, latent_dim=32, label_dim=10):
        super().__init__()
        self.latent_dim = latent_dim
        self.label_dim = label_dim

        # -- ENCODER --------------------------------------------------------
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(1 + label_dim, 32, 3, stride=2, padding=1),   # -> (32, 14, 14)
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),              # -> (64,  7,  7)
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 3, stride=1, padding=1),             # -> (128, 7,  7)
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
        )
        self.fc_mu      = nn.Linear(128 * 7 * 7, latent_dim)
        self.fc_log_var = nn.Linear(128 * 7 * 7, latent_dim)

        # -- DECODER --------------------------------------------------------
        self.decoder_fc = nn.Sequential(
            nn.Linear(latent_dim + label_dim, 128 * 7 * 7),
            nn.LeakyReLU(0.2),
        )
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 3, stride=1, padding=1),    # -> (64,  7,  7)
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),     # -> (32, 14, 14)
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(32, 1, 4, stride=2, padding=1),      # -> (1,  28, 28)
            nn.Sigmoid(),
        )

    # --- TODO 1a: Conditional input ---
    def conditional_input(self, x, label):
        """Concatenate one-hot label (broadcast spatially) with the image."""
        # SOLUTION
        label_map = label.unsqueeze(-1).unsqueeze(-1)                        # (B, 10, 1, 1)
        label_map = label_map.expand(-1, -1, x.size(2), x.size(3))          # (B, 10, 28, 28)
        return torch.cat([x, label_map], dim=1)                              # (B, 11, 28, 28)

    # --- TODO 1b: Reparametrization trick ---
    def reparametrize(self, mu, log_var):
        """Sample z = mu + std * eps, where eps ~ N(0, I)."""
        # SOLUTION
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + std * eps

    # --- TODO 1c: Encode ---
    def encode(self, x, label):
        """Encode image + label -> (mu, log_var)."""
        # SOLUTION
        cond = self.conditional_input(x, label)
        h = self.encoder_conv(cond)
        h = h.flatten(1)
        return self.fc_mu(h), self.fc_log_var(h)

    # --- TODO 1d: Decode ---
    def decode(self, z, label):
        """Decode latent z + label -> reconstructed image."""
        # SOLUTION
        z_cond = torch.cat([z, label], dim=1)                                # (B, latent_dim + 10)
        h = self.decoder_fc(z_cond)
        h = h.view(-1, 128, 7, 7)
        return self.decoder_conv(h)

    # --- TODO 1e: Forward pass ---
    def forward(self, x, label):
        """Full CVAE pass: encode -> reparametrize -> decode."""
        # SOLUTION
        mu, log_var = self.encode(x, label)
        z = self.reparametrize(mu, log_var)
        recon_x = self.decode(z, label)
        return recon_x, mu, log_var

In [ ]:
LATENT_DIM = 32
cvae = ConvCVAE(latent_dim=LATENT_DIM).to(DEVICE)
total_params = sum(p.numel() for p in cvae.parameters())
print(f"ConvCVAE created: {total_params:,} parameters on {DEVICE}")

---
### 🎯 TODO 2 — Complete the CVAE loss function

The CVAE loss has two parts:

🔄 **Reconstruction loss (BCE):** measures pixel-by-pixel how well the output matches the input. Use `F.binary_cross_entropy(recon_x, x, reduction='sum')` divided by batch size.

📏 **KL divergence:** regularises the latent space toward $\mathcal{N}(0, I)$.

$$D_{KL} = -\frac{1}{2} \sum\left(1 + \log\sigma^2 - \mu^2 - \sigma^2\right)$$

**Total loss:** `recon_loss + beta * kl_loss`

In [ ]:
def cvae_loss(recon_x, x, mu, log_var, beta=1.0):
    """
    Compute the CVAE loss = Reconstruction (BCE) + beta * KL divergence.

    Returns
    -------
    total_loss, recon_loss, kl_loss
    """
    batch_size = x.size(0)

    # TODO 2a: Reconstruction loss (BCE, summed over pixels, averaged over batch)
    # SOLUTION
    recon_loss = F.binary_cross_entropy(recon_x, x, reduction='sum') / batch_size

    # TODO 2b: KL divergence (averaged over batch)
    # SOLUTION
    kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp()) / batch_size

    total_loss = recon_loss + beta * kl_loss
    return total_loss, recon_loss, kl_loss

---
### 🏋️ Training the CVAE

We train the CVAE using Adam with cosine learning rate decay. Key hyperparameters:
- `latent_dim = 32` — MNIST is simple enough for a compact latent space
- `beta = 1.0` — standard VAE weighting (you’ll experiment with this later)
- `lr = 1e-3`, `epochs = 20`

In [ ]:
print("Training Conditional VAE...")
cvae, loss_history = train_cvae(cvae, cvae_loss, X_train, y_train_oh,
                                 epochs=20, batch_size=128, lr=1e-3, beta=1.0)

In [ ]:
plot_losses(loss_history)

📊 You should see:
- **Total loss** decreasing overall
- **Reconstruction loss** decreasing steadily (the model learns to reconstruct)
- **KL loss** increasing initially then stabilising (the latent space is being regularised)

---
## 🔍 Part 3: Reconstruction Quality

### How Well Does It Reconstruct?

Let’s pass test images through the CVAE and compare originals vs. reconstructions. Note: CVAE reconstructions may be slightly **blurrier** than a plain autoencoder — this is the price we pay for a well-structured latent space that enables generation.

In [ ]:
recon = reconstruct_cvae(cvae, X_test, y_test_oh)
plot_original_vs_reconstructed(X_test, recon,
                                title="CVAE: Original vs. Reconstructed")

In [ ]:
# @title 📊 Per-digit reconstruction error {display-mode: "form"}
digit_mse, worst_digit = plot_per_digit_mse(recon, X_test, y_test)

### 💬 Debrief

The reconstructions are recognisable but softer than a deterministic autoencoder. This is expected — the KL regularisation prevents the encoder from encoding every tiny detail, which is exactly what makes the latent space smooth enough for generation.

---
## 🎨 Part 4: Conditional Generation — The CVAE Superpower

This is where CVAEs shine ✨. Unlike a regular autoencoder (which can only reconstruct), we can **generate brand-new images of any digit** just by:
1. 🎲 Sampling a random $z$ from $\mathcal{N}(0, I)$
2. 🏷️ Choosing a digit label
3. ➡️ Feeding both to the decoder

> 💼 **Business Analogy:** A regular autoencoder is a **photocopier** — it reproduces what it sees. A CVAE is a **graphic designer** who can create original artwork matching your brief.

---
### 🎯 TODO 3 — Generate digits conditionally

Use `generate_conditional(model, digit, n, latent_dim)` to generate 8 images of specific digits. Try at least 3 different digits and observe the **variety within each class** — each sample has a different style because each $z$ is different.

In [ ]:
# SOLUTION
for digit in [0, 3, 7, 9]:
    generate_conditional(cvae, digit=digit, n=8, latent_dim=LATENT_DIM)

In [ ]:
# @title 🔢 Generate all digits 0-9 in a grid {display-mode: "form"}
generate_digit_grid(cvae, latent_dim=LATENT_DIM, n_per_digit=10)

### 💬 Debrief

Each row should clearly show the **same digit** with **varied styles** (thickness, slant, size). This is the power of conditioning: the label controls *what*, and $z$ controls *how*. If some digits look mixed up, the model might need more training or a different $\beta$.

---
## 🌍 Part 5: Exploring the Latent Space

### What Does the Latent Space Look Like?

In a regular autoencoder, the latent space separates digits into distinct clusters (you saw this in the previous exercise). In a CVAE, the digit identity is carried by the **label**, so the latent space captures **style** instead.

💡 **What to expect:** If the CVAE is well-trained, the latent space should show **more overlap** between digit classes than a plain autoencoder — because $z$ no longer needs to encode *which* digit it is.

In [ ]:
# @title 🌍 Latent space visualisation {display-mode: "form"}
lat_2d, labels_viz = visualise_latent_space_cvae(cvae, X_test, y_test, y_test_oh)

### 💡 How to Read This Plot

Compare this with the latent space from the Autoencoders exercise:
- **Autoencoders**: clear digit clusters (the bottleneck must encode *both* identity and style)
- **CVAE**: more overlap between digits (the label carries identity, $z$ focuses on style)

This overlap is a feature, not a bug — it means the latent space has learned a **shared style representation** across digits.

---
## 🔀 Part 6: Latent Space Interpolation

We can smoothly morph between two images by **interpolating their latent vectors**. Since the CVAE latent space is smooth (thanks to KL regularisation), intermediate points should produce valid-looking images.

---
### 🎯 TODO 4 — Interpolate between two digits

Pick two test images (try different digits). The function `interpolate_latent()` encodes both, linearly interpolates between their $z$ vectors, and decodes each step.

🧪 **Experiment:** Try with `switch_label=False` (fixed label throughout) and `switch_label=True` (label changes at the midpoint). What’s the difference?

In [ ]:
# SOLUTION: Fixed label interpolation
interpolate_latent(cvae, X_test, y_test_oh, idx1=0, idx2=3, n_steps=10)

In [ ]:
# SOLUTION: Label switch at midpoint
interpolate_latent(cvae, X_test, y_test_oh, idx1=0, idx2=3, n_steps=10, switch_label=True)

### 💬 Debrief

- **Fixed label**: the digit identity stays the same, but the *style* gradually changes (you see a smooth morphing of thickness/slant)
- **Label switch**: at the midpoint, the digit identity abruptly changes while the style continues to morph smoothly

This demonstrates the **disentanglement** between content (label) and style ($z$) that the CVAE has learned.

---
## 🎛️ Part 7: The Effect of Beta

$\beta$ controls the balance between reconstruction quality and latent space regularity:

| $\beta$ | Reconstruction | Latent Space | Generation |
|---|---|---|---|
| 🔽 Low (0.1) | Sharp, detailed | Irregular, “holes” | May produce artifacts |
| ⚖️ Standard (1.0) | Good balance | Regular | Decent quality |
| 🔼 High (5.0) | Blurry | Very regular, Gaussian-like | Diverse but fuzzy |

---
### 🎯 TODO 5 — Experiment with different beta values

Train the CVAE with $\beta = 0.1$, $\beta = 1.0$, and $\beta = 5.0$. For each:
1. 🖼️ Generate a digit grid
2. 🔍 Compare: which $\beta$ gives the sharpest images? The most diverse?
3. 📊 Which $\beta$ gives the most regular latent space?

In [ ]:
# SOLUTION
for beta_val in [0.1, 1.0, 5.0]:
    print(f"\n{'='*50}")
    print(f"Training with beta = {beta_val}")
    print(f"{'='*50}")
    model_beta = ConvCVAE(latent_dim=LATENT_DIM).to(DEVICE)
    model_beta, _ = train_cvae(model_beta, cvae_loss, X_train, y_train_oh,
                                epochs=15, batch_size=128, lr=1e-3, beta=beta_val)
    generate_digit_grid(model_beta, latent_dim=LATENT_DIM, n_per_digit=8)

### 💬 Debrief

You should observe:
- **$\beta = 0.1$**: sharp images, but the model may sometimes generate the wrong digit (poor conditioning) ⚠️
- **$\beta = 1.0$**: good balance of sharpness and correct conditioning ✅
- **$\beta = 5.0$**: blurry but very consistent — the latent space is highly regular 🌫️

This trade-off is a fundamental property of VAEs and is known as the **rate-distortion trade-off** in information theory.

---
## 📊 Part 8: How Good Are the Generated Images?

We can quantitatively evaluate generation quality using the same trick as before: train a classifier on real MNIST, then test it on **CVAE-generated images**. If the classifier can recognise them, the generated images are realistic 🤔

In [ ]:
# @title 🧪 Train classifier and evaluate generated images {display-mode: "form"}
print("Training digit classifier on real images...")
classifier = train_classifier(MNISTClassifier, X_train, y_train)

print("\nEvaluating CVAE-generated images...")
acc = evaluate_generated_images(cvae, classifier, latent_dim=LATENT_DIM, n_per_digit=200)

### 💬 Debrief

A well-trained CVAE should achieve **70–90%+ accuracy** on generated images 🎉. The confusion matrix shows which digits are hardest to generate convincingly. Compare this with the reconstruction accuracy from Part 3.

---
## 🏢 Where Are CVAEs Used in the Real World?

| Domain | Application |
|---|---|
| 💊 Drug discovery | Generate molecular structures with desired properties |
| 📈 Data augmentation | Generate labelled training data for underrepresented classes |
| 🧑 Face editing | Change attributes (hair colour, glasses) while preserving identity |
| ✏️ Design | Generate product designs matching a specification |
| 🚨 Anomaly detection | Detect outliers conditioned on expected behaviour |

The original ConditionalVAE project that inspired this exercise used **CelebA** face images with 40 binary attributes — enabling controlled face generation and attribute manipulation.

---

## 🎯 Summary

In this notebook, you’ve built and explored a Conditional Variational Autoencoder:

| Step | What You Did |
|---|---|
| 📦 Data exploration | Loaded MNIST with one-hot digit labels as conditions |
| 🤖 CVAE architecture | Built encoder with spatial label conditioning, decoder with label concatenation |
| 🎯 TODO 1 | Implemented conditional_input, reparametrize, encode, decode, forward |
| 🎯 TODO 2 | Implemented CVAE loss (BCE + β × KL divergence) |
| 🏋️ Training | Trained the CVAE with cosine LR schedule |
| 🔍 Reconstruction | Measured per-digit reconstruction quality |
| 🎯 TODO 3 | Generated specific digits on demand — the CVAE superpower ✨ |
| 🌍 Latent space | Visualised how the CVAE disentangles content from style |
| 🎯 TODO 4 | Interpolated between digits, explored label switching |
| 🎯 TODO 5 | Investigated the β trade-off (sharpness vs. regularity) |
| 📊 Evaluation | Tested generated images with a pre-trained classifier |

### 🔑 Key Takeaways
- **CVAEs** extend VAEs by adding a **condition** (label) at both encoder and decoder
- The latent space captures **style**, while the label encodes **content**
- The **reparametrization trick** enables gradient-based training through sampling
- **$\beta$** controls the reconstruction-regularity trade-off
- CVAEs are a foundation for modern conditional generative models 🚀